# Threads Scraper Analysis Stack 

---

## 架構總覽

```
ScrapeCreators API (/v1/threads/*)
        │  HTTP GET + x-api-key
        ▼
┌─ threads_client.py ──────────────────────┐
│  ThreadsClient._request()   (重試 + 錯誤處理)  │
│  ├─ search_posts()    → list[ThreadsPost]     │
│  ├─ get_profile()     → ThreadsProfile        │
│  ├─ get_user_posts()  → list[ThreadsPost]     │
│  ├─ get_post()        → ThreadsPost           │
│  └─ search_users()    → list[dict]            │
│                                               │
│  ThreadsPost.from_api_response()              │
│  └─ 原始 JSON → 正規化 dataclass               │
└───────────────┬───────────────────────┘
                ▼
┌─ keyword_monitor.py ─────────────────────┐
│  KeywordMonitor                               │
│  ├─ search_keyword()  (去重 by post_id)       │
│  ├─ run_round()       (批次搜多關鍵字)         │
│  └─ export_results()  (CSV + JSON → data/)    │
└───────────────┬───────────────────────┘
                ▼
┌─ Notebook (本檔案) ──────────────────────┐
│  載入 → 語言過濾 → 受眾分群 → 品牌分析        │
└──────────────────────────────────────────┘
```

## Pipeline 節點

| # | 節點 | 說明 |
|---|------|------|
| 1 | 環境設置 | 匯入套件、設定 |
| 2 | API 串接 | 初始化 ThreadsClient、驗證連線 |
| 3 | 爬蟲執行 | 用 KeywordMonitor 批次爬取關鍵字貼文 |
| 4 | 資料 Extract | 檢視 API 原始回傳 → dataclass 解析 → 匯出 |
| 5 | 載入 & 過濾 | 讀取 CSV、繁體中文過濾、清洗 |
| 6 | 受眾輪廓 | 高相關貼文、活躍帳號、關鍵字互動 |
| 7 | 受眾分群 | 求推薦 vs 分享心得、品牌/裝備品類分析 |
| 8 | 洞察總結 | 受眾報告 |

## 節點 1：環境設置

匯入所有需要的套件，包括本專案的 `threads_client` 和 `keyword_monitor`。

In [1]:
import os
import sys
from pathlib import Path
DATA_DIR = Path("data")
# 將 files/ 加入模組搜尋路徑
sys.path.insert(0, str(Path("files").resolve()))
print(f"  - 輸出目錄: {DATA_DIR.resolve()}")

  - 輸出目錄: I:\Fnte Workdir\threads-audience-discovery-stack\jp_car_service\data


In [2]:

import json
import re
import time

from collections import Counter
from datetime import datetime

import pandas as pd
import numpy as np

# 匯入自定模組
from threads_client import (
    ThreadsClient, ThreadsPost, ThreadsProfile,
    posts_to_dicts, save_json, save_csv,
)
from keyword_monitor import KeywordMonitor


In [3]:
from dotenv import load_dotenv
load_dotenv(Path(".env"), override=True)

True

## 節點 2：API 串接

初始化 `ThreadsClient`，驗證 API Key 與連線狀態。

**API 架構：**
- 底層使用 `requests.Session` 保持連線
- 所有請求透過 `_request()` 統一處理：自動重試（429/500）、錯誤分類（401/402/400）
- Header 帶 `x-api-key` 認證

In [4]:
# ── 初始化 API Client ──
# API Key 設定
API_KEY = os.environ.get("SCRAPECREATORS_API_KEY")

client = ThreadsClient(api_key=API_KEY, timeout=30, max_retries=3)
print("✔ ThreadsClient 初始化成功")
print(f"  - Base URL: https://api.scrapecreators.com")
print(f"  - Timeout: 30s, Max retries: 3")

# 驗證連線：查詢剩餘額度
try:
    balance = client.get_credit_balance()
    print(f"  - API 額度剩餘: {balance} credits")
except Exception as e:
    print(f"  - ⚠ 額度查詢失敗: {e}")

✔ ThreadsClient 初始化成功
  - Base URL: https://api.scrapecreators.com
  - Timeout: 30s, Max retries: 3
  - API 額度剩餘: 23870 credits


In [5]:
# # ── 逐一測試 5 個 API Endpoint ──
# # 每次呼叫消耗 1 credit

# print("=" * 55)
# print("  測試 5 個 Threads API Endpoints")
# print("=" * 55)

# # ── Endpoint 1: 搜尋貼文（核心爬蟲 endpoint）──
# print("\n── 1. search_posts('AI') ──")
# print("   GET /v1/threads/search?query=AI")
# test_posts = client.search_posts(query="AI")
# print(f"   回傳 {len(test_posts)} 筆貼文")
# if test_posts:
#     p = test_posts[0]
#     print(f"   範例: @{p.username} | {p.like_count} likes")
#     print(f"   Text: {p.text[:80]}...")

# # ── Endpoint 2: 用戶 Profile ──
# print("\n── 2. get_profile('zuck') ──")
# print("   GET /v1/threads/profile?handle=zuck")
# try:
#     profile = client.get_profile("zuck")
#     print(f"   @{profile.username} | {profile.full_name}")
#     print(f"   Followers: {profile.follower_count:,} | Verified: {profile.is_verified}")
# except Exception as e:
#     print(f"   Error: {e}")

# # ── Endpoint 3: 用戶貼文列表 ──
# print("\n── 3. get_user_posts('zuck') ──")
# print("   GET /v1/threads/user/posts?handle=zuck")
# try:
#     user_posts = client.get_user_posts("zuck")
#     print(f"   回傳 {len(user_posts)} 筆貼文")
# except Exception as e:
#     print(f"   Error: {e}")

# # ── Endpoint 4: 搜尋用戶 ──
# print("\n── 4. search_users('tech') ──")
# print("   GET /v1/threads/search/users?query=tech")
# try:
#     users = client.search_users("tech")
#     print(f"   回傳 {len(users)} 位用戶")
#     for u in users[:3]:
#         print(f"   @{u.get('username', '?')} — {u.get('follower_count', '?')} followers")
# except Exception as e:
#     print(f"   Error: {e}")

# # ── Endpoint 5: 單篇貼文詳情 ──
# print("\n── 5. get_post(url) ──")
# print("   GET /v1/threads/post?url=...")
# if test_posts and test_posts[0].permalink:
#     try:
#         single = client.get_post(test_posts[0].permalink)
#         print(f"   @{single.username} | {single.like_count} likes | {single.reply_count} replies")
#     except Exception as e:
#         print(f"   Error: {e}")

# print("\n" + "=" * 55)

## 節點 3：爬蟲執行

使用 `KeywordMonitor` 批次爬取多個關鍵字的貼文。

**執行流程：**
```
KeywordMonitor.run_round(keywords)
  → 逐一呼叫 client.search_posts(keyword)
    → _request() → GET /v1/threads/search?query=...
    → API JSON → ThreadsPost.from_api_response() 逐筆解析
  → seen_ids 去重（跨關鍵字 + 跨輪次）
  → 累積到 all_posts[keyword]
  → export_results() → CSV + JSON
```

In [6]:
# 5 地點 × 3 意圖關鍵字 + 7 機場/景點接送 = 22 credits / 輪
KEYWORDS = [
    # 日本地點：t1：直接包車意圖
    "日本包車", "沖繩包車", "富士山包車", "大阪包車", "北海道包車", "福岡包車",
    # 日本地點：t2：泛旅遊意圖
    "沖繩自由行", "沖繩旅遊", "日本自由行", "日本旅遊",
    "富士山自由行", "富士山旅遊",
    "大阪自由行", "大阪旅遊",
    "北海道自由行", "北海道旅遊",
    "福岡自由行", "福岡旅遊",
    
    # 韓國地點：t1：直接包車意圖
    "韓國包車", "首爾包車", "釜山包車", "濟州島包車",
    # 韓國地點：t2：泛旅遊意圖
    "韓國自由行", "韓國旅遊",
    "首爾自由行", "首爾旅遊",
    "釜山自由行", "釜山旅遊",
    "濟州島自由行", "濟州島旅遊",
    # t1：機場/景點接送
    # "沖繩機場接送", "那霸機場接送", "羽田機場接送",
    # "關西機場接送", "福岡機場接送", "新千歲機場接送",
    # "成田機場接送",
]
PROJECT_TAG = "japan_korea_charter"

In [7]:
# ── 爬蟲模式設定 ──
# MODE = "range"    → 自訂日期區間（手動回測、一次性分析）
# MODE = "schedule" → 模擬 Airflow 排程時段（測試 DAG 邏輯）
# MODE = "day"      → 單日爬取（快速測試）
MODE = "day"

# --- range 模式參數 ---
RANGE_START = "2026-04-02"
RANGE_END   = "2026-04-06"

# --- day 模式參數 ---
DAY_DATE = "2026-04-16"

# --- schedule 模式參數（模擬排程）---
SIMULATE_HOUR = 10  # 模擬時段: 10, 14, 18

# --- 動態輪數設定 ---
USE_ADAPTIVE = False   # True = run_adaptive (自動多輪), False = 固定輪數
FIXED_ROUNDS = 1       # 僅 USE_ADAPTIVE=False 時生效
MIN_ROUNDS = 2         # 僅 USE_ADAPTIVE=True 時生效
MAX_ROUNDS = 4
DUP_THRESHOLD = 0.80   # 當重複率超過此值，且新增數趨緩時，停止後續輪數（僅 adaptive 模式）

# ── get_scrape_params ──
from datetime import timedelta
from zoneinfo import ZoneInfo

TZ_TPE = ZoneInfo("Asia/Taipei")
SCHEDULE_HOURS = [10, 14, 18]

def get_scrape_params(execution_hour: int, ref_date=None):
    """
    根據排程時段計算 API 日期範圍 + post-filter 時間窗口。
    與 fetch_threads_posts.py 中的邏輯一致。
    """
    from datetime import date
    if ref_date is None:
        ref_date = date.today()

    today_str = ref_date.strftime('%Y-%m-%d')
    yesterday_str = (ref_date - timedelta(days=1)).strftime('%Y-%m-%d')
    ref_dt = datetime.combine(ref_date, datetime.min.time()).replace(tzinfo=TZ_TPE)

    closest_hour = min(SCHEDULE_HOURS, key=lambda h: abs(h - execution_hour))

    if closest_hour == 10:
        return {
            "start_date": yesterday_str,
            "end_date": today_str,
            "time_window": (ref_dt.replace(hour=0), ref_dt.replace(hour=10)),
            "label": f"10:00 slot | API {yesterday_str}~{today_str}, filter 00:00~10:00",
        }
    else:
        idx = SCHEDULE_HOURS.index(closest_hour)
        prev_hour = SCHEDULE_HOURS[idx - 1]
        return {
            "start_date": today_str,
            "end_date": today_str,
            "time_window": (ref_dt.replace(hour=prev_hour), ref_dt.replace(hour=closest_hour)),
            "label": f"{closest_hour}:00 slot | API {today_str}, filter {prev_hour}:00~{closest_hour}:00",
        }

# ── 根據 MODE 決定參數 ──
if MODE == "schedule":
    params = get_scrape_params(SIMULATE_HOUR)
    START_DATE = params["start_date"]
    END_DATE = params["end_date"]
    TIME_WINDOW = params["time_window"]
    print(f"[schedule] {params['label']}")
    print(f"  time_window: {TIME_WINDOW[0].isoformat()} ~ {TIME_WINDOW[1].isoformat()}")
elif MODE == "day":
    START_DATE = DAY_DATE
    END_DATE = DAY_DATE
    TIME_WINDOW = None
    print(f"[day] {DAY_DATE}")
else:  # range
    START_DATE = RANGE_START
    END_DATE = RANGE_END
    TIME_WINDOW = None
    print(f"[range] {RANGE_START} ~ {RANGE_END}")

print(f"  API params: start_date={START_DATE}, end_date={END_DATE}")
print(f"  adaptive: {'ON' if USE_ADAPTIVE else 'OFF'} (rounds={FIXED_ROUNDS if not USE_ADAPTIVE else f'{MIN_ROUNDS}~{MAX_ROUNDS}'})")
# ── 執行關鍵字爬蟲 ──
monitor = KeywordMonitor(client, output_dir=str(DATA_DIR))

print(f"爬取關鍵字: {len(KEYWORDS)} 組")
print(f"專案標記: {PROJECT_TAG}")
print(f"輸出目錄: {DATA_DIR}")
print(f"預計 API 請求數: {len(KEYWORDS)} 次/輪（每關鍵字 1 次）")
print(f"預計 credits 消耗: 約 {len(KEYWORDS)} credits/輪（以 Credits remaining 實際變化為準）\n")

# ── 執行爬蟲 ──
all_round_stats = []

if USE_ADAPTIVE:
    print(f"[adaptive] min={MIN_ROUNDS}, max={MAX_ROUNDS}, dup_threshold={DUP_THRESHOLD}")
    print("[adaptive] 用途：當新貼文趨緩且重複率升高時，自動停止後續輪數，降低 credits 消耗")
    all_posts, all_round_stats = monitor.run_adaptive(
        KEYWORDS,
        start_date=START_DATE,
        end_date=END_DATE,
        min_rounds=MIN_ROUNDS,
        max_rounds=MAX_ROUNDS,
        dup_threshold=DUP_THRESHOLD,
    )
else:
    for round_num in range(FIXED_ROUNDS):
        print(f"{'='*40} 第 {round_num+1}/{FIXED_ROUNDS} 輪 {'='*40}")
        _results, _stats = monitor.run_round(KEYWORDS, start_date=START_DATE, end_date=END_DATE)
        all_round_stats.append(_stats)
        print(f"  API回傳總筆數={_stats['api_total']}, 新增筆數={_stats['new_count']}, 重複率={_stats['dup_ratio']:.1%}")

# ── 合併所有關鍵字，加上 search_keyword 欄，去重 ──
rows = []
for kw, posts in monitor.all_posts.items():
    for p in posts:
        d = posts_to_dicts([p])[0]
        d['search_keyword'] = kw
        rows.append(d)

df_out = pd.DataFrame(rows).drop_duplicates(subset='post_id', keep='first')

# ── 時間窗口 post-filter（僅 schedule 模式）──
if TIME_WINDOW is not None:
    df_out['timestamp'] = pd.to_datetime(df_out['timestamp'], utc=True)
    start_utc = TIME_WINDOW[0].astimezone(ZoneInfo("UTC"))
    end_utc = TIME_WINDOW[1].astimezone(ZoneInfo("UTC"))
    before = len(df_out)
    df_out = df_out[(df_out['timestamp'] >= start_utc) & (df_out['timestamp'] < end_utc)].copy()
    print(f"\n[time_window filter] {before} -> {len(df_out)} posts")

print(f"\n── 爬取完成 ──")
print(f"  去重後貼文: {len(df_out)} 篇")
print(f"  總輪數: {len(all_round_stats)}")
total_api_results = sum(s.get('api_total', 0) for s in all_round_stats)
estimated_credits = len(KEYWORDS) * len(all_round_stats)
print(f"  API回傳總筆數(各輪加總): {total_api_results}")
print(f"  估算 credits 消耗: 約 {estimated_credits}（以 Credits remaining 差值為準）")

# ── 多輪策略覆蓋率提升評估  ── 
round_new = [s.get('new_count', 0) for s in all_round_stats]
if round_new:
    baseline = round_new[0]
    final_cum = sum(round_new)
    lift_pct = ((final_cum - baseline) / baseline * 100) if baseline > 0 else 0.0

    print(f"  第1輪新增: {baseline}")
    print(f"  多輪累積新增: {final_cum}")
    print(f"  覆蓋率提升(相對第1輪): {lift_pct:.1f}%")
    print("  各輪明細:")
    for i, s in enumerate(all_round_stats, 1):
        print(f"    R{i}: api={s['api_total']}, new={s['new_count']}, dup={s['dup_ratio']:.1%}")
#  ── 

# # ── 原始資料立即儲存 ──
# raw_csv = DATA_DIR / f"threads_{PROJECT_TAG}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
# raw_json = raw_csv.with_suffix('.json')
# df_out.to_csv(raw_csv, index=False, encoding='utf-8')
# save_json(df_out.to_dict(orient='records'), str(raw_json))
# print(f"  {raw_csv.name}")
# print(f"  {raw_json.name}")

2026-04-17 10:12:08 [INFO] ─── Search round: 2026-04-17 10:12:08 ───


[day] 2026-04-16
  API params: start_date=2026-04-16, end_date=2026-04-16
  adaptive: OFF (rounds=1)
爬取關鍵字: 30 組
專案標記: japan_korea_charter
輸出目錄: data
預計 API 請求數: 30 次/輪（每關鍵字 1 次）
預計 credits 消耗: 約 30 credits/輪（以 Credits remaining 實際變化為準）

======================================== 第 1/1 輪 ========================================


2026-04-17 10:12:11 [INFO] Credits remaining: 23868
2026-04-17 10:12:11 [INFO]   '日本包車': 10 results, 9 new (total unique: 9)
2026-04-17 10:12:14 [INFO] Credits remaining: 23865
2026-04-17 10:12:14 [INFO]   '沖繩包車': 10 results, 2 new (total unique: 2)
2026-04-17 10:12:18 [INFO] Credits remaining: 23863
2026-04-17 10:12:18 [INFO]   '富士山包車': 10 results, 3 new (total unique: 3)
2026-04-17 10:12:22 [INFO] Credits remaining: 23861
2026-04-17 10:12:22 [INFO]   '大阪包車': 10 results, 7 new (total unique: 7)
2026-04-17 10:12:26 [INFO] Credits remaining: 23860
2026-04-17 10:12:26 [INFO]   '北海道包車': 10 results, 2 new (total unique: 2)
2026-04-17 10:12:29 [INFO] Credits remaining: 23859
2026-04-17 10:12:29 [INFO]   '福岡包車': 10 results, 3 new (total unique: 3)
2026-04-17 10:12:33 [INFO] Credits remaining: 23858
2026-04-17 10:12:33 [INFO]   '沖繩自由行': 10 results, 7 new (total unique: 7)
2026-04-17 10:12:37 [INFO] Credits remaining: 23857
2026-04-17 10:12:37 [INFO]   '沖繩旅遊': 10 results, 7 new (total unique: 

  API回傳總筆數=256, 新增筆數=169, 重複率=34.0%

── 爬取完成 ──
  去重後貼文: 169 篇
  總輪數: 1
  API回傳總筆數(各輪加總): 256
  估算 credits 消耗: 約 30（以 Credits remaining 差值為準）
  第1輪新增: 169
  多輪累積新增: 169
  覆蓋率提升(相對第1輪): 0.0%
  各輪明細:
    R1: api=256, new=169, dup=34.0%


In [8]:
df_out

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,permalink,taken_at,timestamp,search_keyword
0,3876382487937546659_66730426599,anntie122,False,請問脆友有沒有推薦的日本東京包車呢?\n預計6月中過去！車上需要有2個嬰兒座椅\n#東京包車,6,13,0,0,0,False,https://www.threads.net/@anntie122/post/DXLrVW...,1776320824,2026-04-16T06:27:04+00:00,日本包車
1,3876716871341487368_64664896586,aaaaabby.ab,False,大家有推薦的日本包車嗎🥹\n一天富士山、一天鎌倉🙏,6,10,0,0,1,False,https://www.threads.net/@aaaaabby.ab/post/DXM3...,1776360685,2026-04-16T17:31:25+00:00,日本包車
2,3876271987227737801_33936083265,wtf_atwork,False,跪求九州包車大神現身！🙏\n\n預計去九州玩，有兩天需要包車（帶父母共6位大人）：\n\n熊...,4,10,0,0,1,False,https://www.threads.net/@wtf_atwork/post/DXLSN...,1776307651,2026-04-16T02:47:31+00:00,日本包車
3,3876635668919586636_76085302534,chchbiebie,False,有打算年底帶老老少少去日本旅遊\n「老、中、青」三代的旅遊！！\n在猶豫要去沖繩、看富士山、...,0,15,0,0,0,False,https://www.threads.net/@chchbiebie/post/DXMk5...,1776351005,2026-04-16T14:50:05+00:00,日本包車
4,3876526926203110997_63482851033,yu_mei_hung,False,這次沖繩家族旅遊\n6大5小\n選擇「行腳沖繩」13人座\n有小baby所以有加購安全座椅\...,9,6,0,0,0,False,https://www.threads.net/@yu_mei_hung/post/DXMM...,1776338382,2026-04-16T11:19:42+00:00,日本包車
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,3876634274364128664_70041296492,__pydtd,False,濟州島最喜歡的Airbnb🍂,269,2,1,0,144,False,https://www.threads.net/@__pydtd/post/DXMklVPk_2Y,1776350839,2026-04-16T14:47:19+00:00,濟州島旅遊
165,3876578506712401342_65602057060,0123annchan,True,真的別再去排隊吃那些黑豬肉名店了...我怕我說出來會被餐廳老闆罵。🤫\n\n這次在濟州島，我...,29,7,1,0,17,False,https://www.threads.net/@0123annchan/post/DXMX...,1776344191,2026-04-16T12:56:31+00:00,濟州島旅遊
166,3876665092447767239_63607796308,lazyyboy13,False,韓國機機票真的扯！！\n首爾濟州島來回 700 含稅跟行李⋯\n1個小時就能到了 大家真的可...,432,7,12,0,394,False,https://www.threads.net/@lazyyboy13/post/DXMrl...,1776354513,2026-04-16T15:48:33+00:00,濟州島旅遊
167,3876675211115031624_65681943872,nanali816,True,在濟州島的第3天居然失眠了⋯ಥ_ಥ,10,2,0,0,0,False,https://www.threads.net/@nanali816/post/DXMt5C...,1776355719,2026-04-16T16:08:39+00:00,濟州島旅遊


---

## 節點 5：載入資料 & 資料檢視

讀取爬取結果 CSV，執行基礎清洗與資料概覽。

> **語言策略備註：** 本次爬取未使用程式語言過濾。
> 改以關鍵字設計避開日文——使用繁體中文特有字（如「裝備」的「裝」 vs 日文「装」），
> 自然排除多數日文內容。若未來資料出現大量非繁中貼文，可重新啟用語言過濾。

In [9]:
# ── 轉換層 ──
# df_out = pd.read_csv(DATA_DIR / raw_csv.name, encoding='utf-8')

df = df_out.copy()

df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# ── taken_at 轉台灣時間（新增欄位）──
df['taken_at_tpe'] = (
    pd.to_datetime(df['taken_at'], unit='s', utc=True)
      .dt.tz_convert('Asia/Taipei')
      .dt.strftime('%Y-%m-%d %H:%M:%S')
)

# post_date 改以台灣日期為準（讓日期過濾以台灣時區為基準，避免 UTC 跨日誤差）
df['post_date'] = pd.to_datetime(df['taken_at_tpe']).dt.date

df['text_length'] = df['text'].fillna('').str.len()
engagement_cols = ['like_count', 'reply_count', 'repost_count', 'quote_count', 'reshare_count']
df['total_engagement'] = df[engagement_cols].sum(axis=1)

# 日期範圍過濾（post_date 為台灣日期，與 START_DATE/END_DATE 對齊）
from datetime import date
filter_start = date.fromisoformat(START_DATE)
filter_end = date.fromisoformat(END_DATE)
x_df = df[(df['post_date'] < filter_start) | (df['post_date'] > filter_end)].copy()
df = df[(df['post_date'] >= filter_start) & (df['post_date'] <= filter_end)].copy()
print(f"[{MODE} filter] 篩選台灣日期: {START_DATE}" + (f" ~ {END_DATE}" if START_DATE != END_DATE else ""))

# 標記每篇貼文命中了哪些關鍵字
for kw in KEYWORDS:
    df[f'has_{kw}'] = df['text'].fillna('').str.contains(kw, case=False)
df['matched_keywords'] = df[[f'has_{kw}' for kw in KEYWORDS]].sum(axis=1)

# 意圖分類：文中有包車/租車/司機/接送 → t1，其餘 → t2
df['audience_type'] = np.where(
    df['text'].fillna('').str.contains(r'包車|租車|司機|接送', regex=True),
    't1', 't2'
)

print(f"日期篩選後剩餘總篇數: {len(df)}")
print(f"抓到不在設定日期範圍內的貼文: {len(x_df)} 篇")

print(f"  t1 (包車意圖): {(df['audience_type']=='t1').sum()} 篇")
print(f"  t2 (旅遊意圖): {(df['audience_type']=='t2').sum()} 篇")

# ── 地點驗證：篩出不含任何目標地點的貼文 ──
LOCATIONS = ["日本", "沖繩", "富士山", "大阪", "北海道", "福岡", "韓國", "首爾", "釜山", "濟州島"]
location_pattern = '|'.join(LOCATIONS)
has_location = df['text'].fillna('').str.contains(location_pattern, case=False)
only_car_service = df[~has_location].copy()
df = df[has_location].copy()

print(f"── 地點驗證 ──")
print(f"  含目標地點: {len(df)} 篇")
print(f"  不含目標地點 (only_car_service): {len(only_car_service)} 篇")


# ── 商業文判定（COMMERCIAL_TERMS）──
import re
COMMERCIAL_TERMS = [
    "LINE ID", "歡迎諮詢", "招募", "互追", "漲粉",
    "立即預訂", "車隊", "限時", "超時費", "客製",
    "經營", "急單", "服務品質", "客人", "提前預約",
    "免費幫忙", "免費協助", "為您", "領取", "公司",
    "留言", "傳給你",
]
commercial_pattern = '|'.join(COMMERCIAL_TERMS)

def is_commercial(text: str) -> bool:
    return bool(re.search(commercial_pattern, text or '', flags=re.IGNORECASE))


[day filter] 篩選台灣日期: 2026-04-16
日期篩選後剩餘總篇數: 133
抓到不在設定日期範圍內的貼文: 36 篇
  t1 (包車意圖): 32 篇
  t2 (旅遊意圖): 101 篇
── 地點驗證 ──
  含目標地點: 119 篇
  不含目標地點 (only_car_service): 14 篇


In [10]:
x_df

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,permalink,taken_at,timestamp,search_keyword,taken_at_tpe,post_date,text_length,total_engagement
1,3876716871341487368_64664896586,aaaaabby.ab,False,大家有推薦的日本包車嗎🥹\n一天富士山、一天鎌倉🙏,6,10,0,0,1,False,https://www.threads.net/@aaaaabby.ab/post/DXM3...,1776360685,2026-04-16 17:31:25+00:00,日本包車,2026-04-17 01:31:25,2026-04-17,24,17
11,3876718430028759389_70221191637,luleon.12,False,Kkday客路上就很多 挑妳喜歡的就可...舉例【專屬包車】日本東京市區、市區周邊、富士山、...,1,1,0,0,0,True,https://www.threads.net/@luleon.12/post/DXM3t9...,1776360871,2026-04-16 17:34:31+00:00,富士山包車,2026-04-17 01:34:31,2026-04-17,72,2
12,3875766336727812102_63129678923,angela.li1001,False,想請教一下 富士山一日遊，山中湖和河口湖有什麼景點值得去。有沒有去完不會後悔的景，不去會覺得...,981,228,112,2,1381,False,https://www.threads.net/@angela.li1001/post/DX...,1776247373,2026-04-15 10:02:53+00:00,富士山包車,2026-04-15 18:02:53,2026-04-15,52,2704
15,3875940180209682366_63049023851,srsr_0210,False,請問大阪有推薦的機場接送嗎～要去神戶三宮來回🥹,7,8,0,0,1,False,https://www.threads.net/@srsr_0210/post/DXKGw7...,1776268096,2026-04-15 15:48:16+00:00,大阪包車,2026-04-15 23:48:16,2026-04-15,23,16
21,3876827773453690699_63568297902,superspicy235,False,脆上面有沒有泰國包車接送 \n\n車子阿法 或豪華比較新的呢？\n配會講英文司機\n可以代訂...,2,5,0,0,2,False,https://www.threads.net/@superspicy235/post/DX...,1776373906,2026-04-16 21:11:46+00:00,北海道包車,2026-04-17 05:11:46,2026-04-17,49,9
33,3876690134549734295_63447288982,xuanee24,False,又回到了沖繩話題了\n因為要去的時間是六月底\n想到天氣很炎熱的問題\n有什麼很厲害的涼感防...,1,3,0,0,0,False,https://www.threads.net/@xuanee24/post/DXMxSNG...,1776357498,2026-04-16 16:38:18+00:00,沖繩旅遊,2026-04-17 00:38:18,2026-04-17,135,4
41,3876687221420049353_34245718915,leonardo__hi,False,真心提問🙆‍♂️\n新手小白嘗試安排自由行，\n4位成人10-11月東京5日遊🇯🇵，\n請問...,12,32,0,0,7,False,https://www.threads.net/@leonardo__hi/post/DXM...,1776357151,2026-04-16 16:32:31+00:00,日本自由行,2026-04-17 00:32:31,2026-04-17,106,51
44,3876901753686322056_64218900278,ajina0933,False,Ajina日本自由行\n福島夏井千本桜🌸🌸🌸,27,2,0,0,0,False,https://www.threads.net/@ajina0933/post/DXNhZq...,1776382725,2026-04-16 23:38:45+00:00,日本自由行,2026-04-17 07:38:45,2026-04-17,21,29
46,3876881349705887365_63395051124,jill_chchen,False,日本自由行\n第一晚就在超商隨便吃吃\n然後自己喝酒去了哈哈哈哈\n\n超商買了兩樣東西都超...,21,0,1,0,3,False,https://www.threads.net/@jill_chchen/post/DXNc...,1776380293,2026-04-16 22:58:13+00:00,日本自由行,2026-04-17 06:58:13,2026-04-17,168,25
63,3599728490733591594_63453899128,jessielin_0901,False,長大後第一次的北海道🇯🇵\n喜歡自由行的我心血來潮做了𝟭本旅遊手冊📚\n完成的時候成就感爆棚...,0,1266,0,0,0,False,https://www.threads.net/@jessielin_0901/post/D...,1743341416,2025-03-30 13:30:16+00:00,北海道自由行,2025-03-30 21:30:16,2025-03-30,156,1266


In [11]:
# 排除商家文
df['is_commercial'] = df['text'].fillna('').apply(is_commercial)
post_is_commercial = df[df['is_commercial']].copy()
df_clean = df[~df['is_commercial']].copy()

print(f"商家文過濾: {len(df)} → {len(df_clean)} 篇（排除 {len(post_is_commercial)} 篇商家文）")

商家文過濾: 119 → 103 篇（排除 16 篇商家文）


In [12]:
post_is_commercial

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,has_韓國旅遊,has_首爾自由行,has_首爾旅遊,has_釜山自由行,has_釜山旅遊,has_濟州島自由行,has_濟州島旅遊,matched_keywords,audience_type,is_commercial
5,3876490944955276656_77222713373,gtjapantour,False,🚗 日本包車｜機場接送｜一日包車｜多日訂製 \n不想自己查車次、搬行李、趕電車，交給我們就好...,1,2,0,0,0,False,...,False,False,False,False,False,False,False,3,t1,True
6,3876392380387108586_41432450355,attomitravel.jp,False,🚗給大家科普一下｜來日本包車為什麼一定要選正规「綠牌車」？\n\n很多人來日本包車，其實不知...,3,2,1,0,3,False,...,False,False,False,False,False,False,False,2,t1,True
7,3876654802973957846_63645440554,namagomi12,False,這次坐自己公司的車子來福岡看辦公室地點\n真的非常舒適…\n日本旅遊包車一生推..,1,0,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,True
9,3876446058024901311_63283640521,_____bao.312,False,📍5/25-5/28 沖繩旅遊\n想詢問包車推薦🚗\n\n✔ 人數：6大2小（2個安全座椅）...,3,4,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,True
18,3876596647313232575_63386601006,lineage8573,False,心が温まるワンシーンでした。\n暖心的一幕\n\n🎏【大阪京都和服外拍，和服體驗，日本一般旅...,1,0,0,0,0,False,...,False,False,False,False,False,False,False,0,t1,True
47,3876438918984623960_74656012196,risin._,False,請問大家去日本有推薦的計程車車隊或者是app之類的嗎？\n或者是日本計程車有哪些是需要特別注...,10,11,0,0,0,False,...,False,False,False,False,False,False,False,0,t2,True
66,3876282620812848969_33973516366,lucky8085888,False,🇯🇵北海道新開🔥人氣日料｜徵台灣、香港創作者\n最近要來北海道的先看這篇👇\n直接請你吃😆\...,0,0,0,0,0,False,...,False,False,False,False,False,False,False,2,t2,True
82,3876604949417841889_73692190051,ruru.next_trip,False,九州自由行旅遊｜初心者攻略\n\nPART 1｜九州的歷史\n\n1️⃣ 古代九州｜對外窗口...,2,0,0,0,0,False,...,False,False,False,False,False,False,False,0,t2,True
83,3876628347678318834_67470214204,lioupeijyun,False,卡五一勞動節唷！【九州雙城｜福岡熊本】\n福岡及熊本飯店8天7夜自由行(含稅) \nhttp...,10,0,0,0,0,False,...,False,False,False,False,False,False,False,0,t2,True
92,3876268300678841684_80057577789,longshao_0722,False,來韓國玩想輕鬆逛街不累？\n專車接送＋購物包車一次全包！\n \n✈️ 仁川／金浦機場專車接...,4,4,0,0,16,False,...,False,False,False,False,False,False,False,1,t1,True


In [13]:
only_car_service

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,has_韓國自由行,has_韓國旅遊,has_首爾自由行,has_首爾旅遊,has_釜山自由行,has_釜山旅遊,has_濟州島自由行,has_濟州島旅遊,matched_keywords,audience_type
10,3876638343434481714_63133591958,yichen0113,False,第一次家族旅遊～\n參加成員有：\n公公（坐輪椅）需有無障礙住宿、婆婆、小叔*2、婆婆友人\...,18,19,0,0,1,False,...,False,False,False,False,False,False,False,False,0,t1
16,3876249989404003790_63365067996,austinwei0717,False,週四大家早安👋🏻☀️\n小弟在此宣布📣\n𝐓𝐡𝐫𝐞𝐚𝐝𝐬《想被看見的帳號》第🔟屆\n正式開放...,143,307,4,1,2,False,...,False,False,False,False,False,False,False,False,0,t2
22,3876519186779961236_70627695524,tzu_hsuan_chang,False,徵求司機以及包車，朋友跟他媽媽要來台北玩四天，想包四天的車，預算10000-15000直間，...,20,46,0,0,10,False,...,False,False,False,False,False,False,False,False,0,t1
25,3876593943086453632_77462381556,kyushu_tule_taxi,False,今日 接送客人\n界 由佈院 \n很舒服的一個飯店 環境真是不錯,2,0,0,0,1,False,...,False,False,False,False,False,False,False,False,0,t1
49,3876550073610022764_68047138666,cca.na16,False,去東京旅遊應該要直接買esim還是漫遊？\n威訊的esim有推薦嗎？,8,22,0,0,6,False,...,False,False,False,False,False,False,False,False,0,t2
60,3876244902375962856_67628054741,2n3dholiday,True,如果你跟我一樣\n超級討厭去關西機場\n（因為人好多 離市區好遠 轉車好麻煩）\n那建議你試...,949,72,80,5,2059,False,...,False,False,False,False,False,False,False,False,0,t2
85,3876555639955201658_64699230974,j3830377,False,聽說脆友很厲害\n下個月要去九州旅遊\n想請問大家有推薦不吃牛的人也可以吃的燒肉店嗎～\n爬...,4,1,0,0,0,False,...,False,False,False,False,False,False,False,False,0,t2
86,3876361531006030099_68859774263,eichikuychu,False,出國第一天原sim卡就不見😭,3,9,0,0,0,False,...,False,False,False,False,False,False,False,False,0,t2
88,3876320924161845174_66516639656,adamnien,False,在太宰府抽籤的時候\n撿到一個錢包\n裡面有兩本護照\n因為很重要\n怕放在原地會不見\n我...,578,44,45,2,2,False,...,False,False,False,False,False,False,False,False,0,t2
97,3876323165949499311_31981072356,taikingtourgo,False,包車真的很讚，之前出國試過一次包車整個回不去，不用自己查電車怎麼搭，也不用提著大包小包搭電車...,0,0,0,0,0,False,...,False,False,False,False,False,False,False,False,0,t1


In [14]:
df_clean

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,has_韓國旅遊,has_首爾自由行,has_首爾旅遊,has_釜山自由行,has_釜山旅遊,has_濟州島自由行,has_濟州島旅遊,matched_keywords,audience_type,is_commercial
0,3876382487937546659_66730426599,anntie122,False,請問脆友有沒有推薦的日本東京包車呢?\n預計6月中過去！車上需要有2個嬰兒座椅\n#東京包車,6,13,0,0,0,False,...,False,False,False,False,False,False,False,0,t1,False
2,3876271987227737801_33936083265,wtf_atwork,False,跪求九州包車大神現身！🙏\n\n預計去九州玩，有兩天需要包車（帶父母共6位大人）：\n\n熊...,4,10,0,0,1,False,...,False,False,False,False,False,False,False,0,t1,False
3,3876635668919586636_76085302534,chchbiebie,False,有打算年底帶老老少少去日本旅遊\n「老、中、青」三代的旅遊！！\n在猶豫要去沖繩、看富士山、...,0,15,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,False
4,3876526926203110997_63482851033,yu_mei_hung,False,這次沖繩家族旅遊\n6大5小\n選擇「行腳沖繩」13人座\n有小baby所以有加購安全座椅\...,9,6,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,False
8,3876438198645239298_67299142909,chenpig0314,False,有沒有推薦沖繩包車\n10大2小目前還沒有安排行程\n想了解一下包車費用\n謝謝☺️☺️☺️,6,10,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,3876489006809031623_63241634733,tin_en,False,如果你有去濟州島西歸浦偶來市場，\n推薦來吃這家鴨胸～半份才12000韓元～\n肉量還蠻多的...,157,4,10,0,314,False,...,False,False,False,False,False,False,False,0,t2,False
163,3876469885186817066_64911664206,aeeee_ouo,False,沒買過去濟州島的機票，想詢問：\n虎航11月初桃園機場來回濟州島六天五夜，\n（去程班機06...,16,17,0,0,20,False,...,False,False,False,False,False,False,False,0,t2,False
164,3876634274364128664_70041296492,__pydtd,False,濟州島最喜歡的Airbnb🍂,269,2,1,0,144,False,...,False,False,False,False,False,False,False,0,t2,False
166,3876665092447767239_63607796308,lazyyboy13,False,韓國機機票真的扯！！\n首爾濟州島來回 700 含稅跟行李⋯\n1個小時就能到了 大家真的可...,432,7,12,0,394,False,...,False,False,False,False,False,False,False,0,t2,False


In [15]:
# ── 資料觀察 ──
print("=== 資料概覽 ===")
print(f"  貼文數: {len(df_clean)}")
print(f"  不重複帳號: {df_clean['username'].nunique()}")
print(f"  時間範圍: {df_clean['post_date'].min():%Y-%m-%d} ~ {df_clean['post_date'].max():%Y-%m-%d}")
print(f"  is_reply 分布: {df_clean['is_reply'].value_counts().to_dict()}")

print(f"\n=== search_keyword 來源分布 ===")
print(df_clean['search_keyword'].value_counts().to_string())

print(f"\n=== 互動指標描述統計 ===")
print(df_clean[engagement_cols + ['total_engagement']].describe().round(1).to_string())

=== 資料概覽 ===
  貼文數: 103
  不重複帳號: 102
  時間範圍: 2026-04-16 ~ 2026-04-16
  is_reply 分布: {False: 102, True: 1}

=== search_keyword 來源分布 ===
search_keyword
北海道旅遊     8
濟州島旅遊     7
沖繩自由行     7
釜山旅遊      6
沖繩旅遊      6
韓國旅遊      6
日本旅遊      6
韓國自由行     6
首爾旅遊      5
首爾包車      5
日本包車      5
大阪旅遊      5
日本自由行     4
釜山自由行     4
大阪包車      4
福岡旅遊      3
韓國包車      3
濟州島自由行    3
大阪自由行     2
福岡包車      2
福岡自由行     2
北海道自由行    1
富士山包車     1
釜山包車      1
首爾自由行     1

=== 互動指標描述統計 ===
       like_count  reply_count  repost_count  quote_count  reshare_count  total_engagement
count       103.0        103.0         103.0        103.0          103.0             103.0
mean        459.1         17.2          65.9          4.7          338.5             885.4
std        3380.7         84.3         603.1         44.5         1969.7            6062.1
min           0.0          0.0           0.0          0.0            0.0               0.0
25%           3.0          1.0           0.0          0.0            0.0     

In [16]:
# # ── 輸出清洗後 CSV/JSON ──
# csv_path = DATA_DIR / f"threads_{PROJECT_TAG}_clean_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
# json_path = csv_path.with_suffix('.json')

# df_clean.to_csv(csv_path, index=False, encoding='utf-8')
# save_json(df_clean.to_dict(orient='records'), str(json_path))

# print(f"✔ {csv_path.name}")
# print(f"✔ {json_path.name}")

In [17]:
# ── 上傳到 Google Sheets ──
import gspread
from google.oauth2.service_account import Credentials

# 篩選欄位（與 vn_visa 對齊；jp_car 不含 post_category）
ouput_df = df_clean[[
    'username', 'taken_at_tpe', 'post_date', 'text',
    'like_count', 'reply_count', 'repost_count', 'quote_count', 'reshare_count',
    'permalink', 'search_keyword', 'audience_type',
]].copy()
ouput_df['post_category'] = ''
ouput_df['scrape_date'] = datetime.now().strftime('%Y-%m-%d')

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
CREDS_PATH = Path("ads-de-01-757fc521ef01.json")
SHEET_ID = "1ld3Mt6ZbT1uNSidxYZN94EuIaHL8AIIw2gNZTiFH4Jw"

creds = Credentials.from_service_account_file(CREDS_PATH, scopes=SCOPES)
gc = gspread.authorize(creds)

sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1

# ── 安全寫入：明確定位最後一行，不依賴 append_rows 的表格偵測 ──
existing = ws.get_all_values()
next_row = len(existing) + 1

if not existing or not any(str(cell).strip() for cell in existing[0]):
    ws.update(range_name='A1', values=[ouput_df.columns.tolist()], value_input_option="RAW")
    next_row = 2
    print(f"[init] 已寫入標題列")

data_rows = ouput_df.astype(str).values.tolist()
ws.update(range_name=f'A{next_row}', values=data_rows, value_input_option="USER_ENTERED")

print(f"[OK] 已寫入 {len(data_rows)} 筆到 A{next_row}~A{next_row + len(data_rows) - 1}")
print(f"  Sheet 目前共 {next_row + len(data_rows) - 1} 行（含標題）")
print(f"  https://docs.google.com/spreadsheets/d/{SHEET_ID}")

[OK] 已寫入 103 筆到 A967~A1069
  Sheet 目前共 1069 行（含標題）
  https://docs.google.com/spreadsheets/d/1ld3Mt6ZbT1uNSidxYZN94EuIaHL8AIIw2gNZTiFH4Jw


In [18]:
df_clean

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,has_韓國旅遊,has_首爾自由行,has_首爾旅遊,has_釜山自由行,has_釜山旅遊,has_濟州島自由行,has_濟州島旅遊,matched_keywords,audience_type,is_commercial
0,3876382487937546659_66730426599,anntie122,False,請問脆友有沒有推薦的日本東京包車呢?\n預計6月中過去！車上需要有2個嬰兒座椅\n#東京包車,6,13,0,0,0,False,...,False,False,False,False,False,False,False,0,t1,False
2,3876271987227737801_33936083265,wtf_atwork,False,跪求九州包車大神現身！🙏\n\n預計去九州玩，有兩天需要包車（帶父母共6位大人）：\n\n熊...,4,10,0,0,1,False,...,False,False,False,False,False,False,False,0,t1,False
3,3876635668919586636_76085302534,chchbiebie,False,有打算年底帶老老少少去日本旅遊\n「老、中、青」三代的旅遊！！\n在猶豫要去沖繩、看富士山、...,0,15,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,False
4,3876526926203110997_63482851033,yu_mei_hung,False,這次沖繩家族旅遊\n6大5小\n選擇「行腳沖繩」13人座\n有小baby所以有加購安全座椅\...,9,6,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,False
8,3876438198645239298_67299142909,chenpig0314,False,有沒有推薦沖繩包車\n10大2小目前還沒有安排行程\n想了解一下包車費用\n謝謝☺️☺️☺️,6,10,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,3876489006809031623_63241634733,tin_en,False,如果你有去濟州島西歸浦偶來市場，\n推薦來吃這家鴨胸～半份才12000韓元～\n肉量還蠻多的...,157,4,10,0,314,False,...,False,False,False,False,False,False,False,0,t2,False
163,3876469885186817066_64911664206,aeeee_ouo,False,沒買過去濟州島的機票，想詢問：\n虎航11月初桃園機場來回濟州島六天五夜，\n（去程班機06...,16,17,0,0,20,False,...,False,False,False,False,False,False,False,0,t2,False
164,3876634274364128664_70041296492,__pydtd,False,濟州島最喜歡的Airbnb🍂,269,2,1,0,144,False,...,False,False,False,False,False,False,False,0,t2,False
166,3876665092447767239_63607796308,lazyyboy13,False,韓國機機票真的扯！！\n首爾濟州島來回 700 含稅跟行李⋯\n1個小時就能到了 大家真的可...,432,7,12,0,394,False,...,False,False,False,False,False,False,False,0,t2,False


In [ ]:
backup_csv_path = DATA_DIR / f"threads_car_service_post{datetime.now().strftime('%Y%m%d')}.csv"
ouput_df.to_csv(backup_csv_path, index=False, encoding='utf-8-sig')
print(f"[OK] 已備份 CSV: {backup_csv_path}")